In [4]:
"""
ResNet-56 on CIFAR-10, trained from scratch (no pretrained weights).
Target: ~92.5-93.5% test accuracy.

Architecture follows He et al. (2015), "Deep Residual Learning", Sec. 4.2:
  3 stages x 9 basic blocks (16/32/64 channels) -> 6n+2 = 56 layers, ~0.85M params.

Usage:
    pip install torch torchvision
    python train_resnet56_cifar10.py                 # default 200 epochs
    python train_resnet56_cifar10.py --epochs 180 --batch-size 128
"""

import argparse
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T


# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
class LambdaLayer(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x):
        return self.fn(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            # Option A from the paper: parameter-free shortcut
            # (subsample spatially + zero-pad the extra channels)
            pad = (planes - in_planes) // 2
            self.shortcut = LambdaLayer(
                lambda x: F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, pad, pad))
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetCIFAR(nn.Module):
    def __init__(self, depth=56, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "depth must be 6n+2"
        n = (depth - 2) // 6
        self.in_planes = 16

        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, n, stride=1)
        self.layer2 = self._make_layer(32, n, stride=2)
        self.layer3 = self._make_layer(64, n, stride=2)
        self.fc = nn.Linear(64, num_classes)

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        # Zero-init the last BN in each residual branch -> each block starts
        # as identity; helps deep nets train more stably.
        for m in self.modules():
            if isinstance(m, BasicBlock):
                nn.init.zeros_(m.bn2.weight)

    def _make_layer(self, planes, blocks, stride):
        layers = []
        for s in [stride] + [1] * (blocks - 1):
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer3(self.layer2(self.layer1(out)))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def resnet56(num_classes=10):
    return ResNetCIFAR(depth=56, num_classes=num_classes)


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def get_loaders(data_dir, batch_size, workers):
    mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

    train_set = torchvision.datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
    test_set = torchvision.datasets.CIFAR10(data_dir, train=False, download=True, transform=test_tf)

    pin = torch.cuda.is_available()
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=workers,
        pin_memory=pin, drop_last=True, persistent_workers=workers > 0)
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=512, shuffle=False, num_workers=workers,
        pin_memory=pin, persistent_workers=workers > 0)
    return train_loader, test_loader


# ----------------------------------------------------------------------------
# Train / eval
# ----------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, use_amp):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # per-iteration schedule (warmup + cosine)

        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_amp):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(x)
            loss = criterion(out, y)
        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, 100.0 * correct / total


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--epochs", type=int, default=150)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--momentum", type=float, default=0.9)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--warmup-epochs", type=int, default=2)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--data-dir", type=str, default="./data")
    p.add_argument("--save", type=str, default="resnet56_cifar10_best.pth")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no-amp", action="store_true", help="disable mixed precision")
    args, unknown = p.parse_known_args()
    #args = p.parse_args()

    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_amp
    torch.backends.cudnn.benchmark = True
    print(f"Device: {device} | AMP: {use_amp}")

    train_loader, test_loader = get_loaders(args.data_dir, args.batch_size, args.workers)

    model = resnet56().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"ResNet-56 parameters: {n_params / 1e6:.3f}M")

    criterion = nn.CrossEntropyLoss()
    # Don't apply weight decay to BN params and biases
    decay, no_decay = [], []
    for name, prm in model.named_parameters():
        (no_decay if prm.ndim <= 1 else decay).append(prm)
    optimizer = torch.optim.SGD(
        [{"params": decay, "weight_decay": args.wd},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=args.lr, momentum=args.momentum, nesterov=True)

    steps_per_epoch = len(train_loader)
    total_steps = args.epochs * steps_per_epoch
    warmup_steps = args.warmup_epochs * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps),
        ],
        milestones=[warmup_steps],
    )
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    best_acc = 0.0
    header = f"{'Epoch':>5} | {'LR':>8} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Loss':>9} | {'Test Acc':>8} | {'Best':>6} | {'Time':>6}"
    print(header)
    print("-" * len(header))

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer,
                                          scheduler, scaler, device, use_amp)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device, use_amp)

        if te_acc > best_acc:
            best_acc = te_acc
            torch.save({"epoch": epoch, "model": model.state_dict(), "test_acc": te_acc}, args.save)

        print(f"{epoch:>5} | {lr:>8.5f} | {tr_loss:>10.4f} | {tr_acc:>8.2f}% | "
              f"{te_loss:>9.4f} | {te_acc:>7.2f}% | {best_acc:>5.2f}% | {time.time() - t0:>5.1f}s",
              flush=True)

    print(f"\nFinal test accuracy: {te_acc:.2f}% | Best test accuracy: {best_acc:.2f}%")
    print(f"Best checkpoint saved to: {args.save}")


if __name__ == "__main__":
    main()

Device: cuda | AMP: True
Using downloaded and verified file: ./data/cifar-10-python.tar.gz
Extracting ./data/cifar-10-python.tar.gz to ./data


/usr/lib64/python3.11/tarfile.py:2303: RuntimeWarning: The default behavior of tarfile extraction has been changed to disallow common exploits (including CVE-2007-4559). By default, absolute/parent paths are disallowed and some mode bits are cleared. See https://access.redhat.com/articles/7004769 for more details.
  warnings.warn(


Files already downloaded and verified
ResNet-56 parameters: 0.853M
Epoch |       LR | Train Loss | Train Acc | Test Loss | Test Acc |   Best |   Time
----------------------------------------------------------------------------------
    1 |  0.01000 |     1.8076 |    32.70% |    1.8498 |   38.83% | 38.83% |  12.9s


/home/mtech/2025/poulami.mukherjee25m/torchgpu_env/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


    2 |  0.05500 |     1.2352 |    55.23% |    1.7591 |   46.61% | 46.61% |  11.6s
    3 |  0.10000 |     0.9929 |    64.75% |    1.3930 |   57.51% | 57.51% |  11.7s
    4 |  0.09999 |     0.8439 |    70.42% |    1.5363 |   54.28% | 57.51% |  12.0s
    5 |  0.09995 |     0.7471 |    73.88% |    1.1789 |   65.66% | 65.66% |  11.4s
    6 |  0.09990 |     0.6866 |    76.25% |    0.9723 |   70.54% | 70.54% |  11.4s
    7 |  0.09982 |     0.6516 |    77.56% |    1.4551 |   58.30% | 70.54% |  11.9s
    8 |  0.09972 |     0.6214 |    78.61% |    1.0408 |   67.73% | 70.54% |  11.7s
    9 |  0.09960 |     0.5982 |    79.67% |    0.8027 |   73.12% | 73.12% |  11.4s
   10 |  0.09945 |     0.5852 |    79.83% |    1.3104 |   64.94% | 73.12% |  11.4s
   11 |  0.09928 |     0.5670 |    80.48% |    0.8872 |   72.36% | 73.12% |  11.3s
   12 |  0.09909 |     0.5538 |    81.01% |    0.7516 |   75.63% | 75.63% |  11.8s
   13 |  0.09888 |     0.5383 |    81.62% |    0.7198 |   76.40% | 76.40% |  11.7s
   1

In [2]:
%cd /home/mtech/2025/poulami.mukherjee25m/4thSem/resnet56/

/home/mtech/2025/poulami.mukherjee25m/4thSem/resnet56
